In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import glob
import shelve
import pickle
import numpy as np
import torch
import pandas as pd

from datetime import timedelta

from torch.nn import functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import json
from raven.model import GPTConfig, GPT
from tqdm import tqdm


from raven.dataset import UnifiedSeqEHRDataset

In [ ]:
# ── Configurable paths ──
NYU_DATA_DIR = "./data"  # Path to NYU processed data
NYU_PROCESSED_DIR = os.path.join(NYU_DATA_DIR, "processed_parts_final_v4")
EHRSHOT_DIR = "./ehrshot_data"  # Path to EHRSHOT dataset
EHRSHOT_MEDS_DIR = os.path.join(EHRSHOT_DIR, "EHRSHOT_MEDS")
EHRSHOT_ASSETS_DIR = os.path.join(EHRSHOT_DIR, "EHRSHOT_ASSETS")
MAPPING_DIR = "./mapping"  # Path to code mapping files
CAPSTONE_DIR = "./data"  # Path to processed data
MODEL_OUT_DIR = "./output"  # Path to model checkpoints

In [ ]:
final_header = os.path.join(NYU_PROCESSED_DIR, 'data_files/filtered_headers_token_merge.json')
# need this for converding indices to tokens
with open(final_header, 'rb') as file:
    final_headers = json.load(file)
header_to_ind = {key: i for i, key in enumerate(final_headers) }

ind_to_header = {i:key for i, key in enumerate(final_headers)}


In [ ]:
len(header_to_ind)

In [ ]:
label_df = pd.read_parquet(os.path.join(EHRSHOT_MEDS_DIR, 'labels/new_pancan/labels.parquet'))
# data_df = pd.read_parquet(os.path.join(EHRSHOT_MEDS_DIR, 'data/data.parquet'))


In [ ]:
label_df

In [ ]:
ehrshotcsv = pd.read_csv(os.path.join(EHRSHOT_ASSETS_DIR, 'data/ehrshot.csv'))


In [ ]:
ehrshotcsv

In [ ]:
ehrshotcsv[ehrshotcsv['code'].str.startswith('Domain')]

In [ ]:
# in the code column, extract all the unique types of code and the count of each type, the type is indicated by the string before the first /
split_codes = ehrshotcsv['code'].str.split('/')
split_codes.str[0].value_counts()
# group by each type and count the number of unique subtypes in each type
ehrshotcsv['type'] = split_codes.str[0]
ehrshotcsv.groupby('type')['code'].nunique()

In [ ]:
split_codes.str[0].value_counts()

In [ ]:
# for ehrshotcsv, drop the column unnamed, end, omop_table
ehrshotcsv = ehrshotcsv.drop(columns=['Unnamed: 0', 'end', 'omop_table', 'type'], errors='ignore')

# remove the rows that have code starting with CARE_SITE, ICD10PCS
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('CARE_SITE')] # care sites
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('CPT4')] # procedures
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('ICD10PCS')] # procedures
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('ICD9Proc')] # procedures
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('CVX')] # vaccines
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Visit')] # visit types
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('HCPCS')] # medical equipment
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Domain')]
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Medicare Specialty')]
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('RxNorm Extension')] # RxNorm Extension/OMOP2028607
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('CMS Place of Service')]
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Cancer Modifier')] # cancer staging
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('OMOP Extension')]
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Condition Type')]


In [ ]:
ehrshotcsv

In [ ]:
ehrshotcsv['code'].str.split('/').str[0].value_counts()

In [ ]:
# write csv to 
ehrshotcsv.to_csv('EHRSHOT_Data/ehrshot_cleaned.csv', index=False)

In [ ]:
# load cleaned csv
ehrshotcsv = pd.read_csv('EHRSHOT_Data/ehrshot_cleaned.csv')

# Translating Code Names

In [ ]:
# Initialize the new column with NA
ehrshotcsv['target_token_id'] = pd.NA

In [ ]:
ehrshotcsv

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

## Race, Ethinicity, and Gender

In [ ]:
# Define the translation map
# Add more mappings to this dictionary as needed
ehrshot_to_target_map = {
    # Race 1 American Indian does not exist in target
    "Race/2": "demographics_RACE_Asian",
    "Race/3": "demographics_RACE_African American (Black)",
    "Race/4": "demographics_RACE_Native Hawaiian or Other Pacific Islander",
    "Race/5": "demographics_RACE_White",
    # no code from ehrshot for unknown race
    "Ethnicity/Hispanic": "demographics_ETHNICITY_Spanish/Hispanic Origin",
    "Ethnicity/Not Hispanic" : "demographics_ETHNICITY_Not of Spanish/Hispanic Origin",
    "Gender/F": "demographics_GENDER_Female",
    "Gender/M": "demographics_GENDER_Male"
}

In [ ]:
# Apply the translation only for rows with codes in our map
# Find rows that need translation
mask = ehrshotcsv['code'].isin(ehrshot_to_target_map.keys())

# Get the target code string (e.g., "demographics_RACE_Asian")
target_codes_series = ehrshotcsv.loc[mask, 'code'].map(ehrshot_to_target_map)

# Map the target code string to the token ID using header_to_ind
# codes not in header_to_ind will become NaN
target_token_ids = target_codes_series.map(header_to_ind)

# Assign the token IDs to the new column
ehrshotcsv.loc[mask, 'target_token_id'] = target_token_ids

# Show the results for verification
print(f"Updated {mask.sum()} rows.")
print(ehrshotcsv.loc[mask, ['code', 'target_token_id']])

In [ ]:
ehrshotcsv

## Medications

In [ ]:
nyu_medication_mapping = pd.read_csv(os.path.join(MAPPING_DIR, "NDC_ATC_RXNORM_to_md_name_20251007.csv"))
nyu_medication_mapping.drop("Unnamed: 0", axis=1, inplace=True)


In [ ]:
nyu_medication_mapping

In [ ]:
# for resolving conflicts when multiple tokens can be mapped
from collections import Counter
import shelve
from tqdm import tqdm

# Load training data info
train_df_path = os.path.join(NYU_PROCESSED_DIR, "csvs/train.csv")
train_df = pd.read_csv(train_df_path)

db_folder = NYU_PROCESSED_DIR
# Open shelves
db_dict = {part: shelve.open(db_folder+'/processed_part_'+str(part)+'.shelve') for part in tqdm(range(40), desc="Opening shelves")}

# Load mask
remove_inds = np.load(os.path.join(NYU_PROCESSED_DIR, "data_files/delete_inds.npy"))
total_prev_tokens = 57735
keep_mask = np.ones(total_prev_tokens, dtype=bool)
keep_mask[remove_inds] = False

# Count token frequencies
print("Counting token frequencies in training data...")
token_counts = Counter()

# Iterate by part for efficiency
for part_id in range(40):
    # Filter patients in this part
    part_patients = train_df[train_df['part'] == part_id]['patient_id']
    
    if len(part_patients) == 0:
        continue
        
    shelve_file = db_dict[part_id]
    
    for pid in tqdm(part_patients, desc=f"Part {part_id}", leave=False):
        try:
            # Assuming patient_id in CSV matches key in shelve
            p_key = str(pid) 
            if p_key in shelve_file:
                a = shelve_file[p_key]
                # Apply mask (columns)
                a_reduced = a[:, keep_mask]
                
                # Get non-zero indices (tokens)
                if hasattr(a_reduced, "nonzero"):
                     rows, token_arr = a_reduced.nonzero()
                else:
                     rows, token_arr = np.nonzero(a_reduced)

                # Convert to simple list/array and update
                token_counts.update(token_arr)
        except Exception as e:
            continue

print(f"Finished counting. Found {len(token_counts)} unique tokens.")


In [ ]:
# save token counts dict
with open('EHRSHOT_Data/token_counts.pkl', 'wb') as f:
    import pickle
    pickle.dump(token_counts, f)


In [ ]:
# 1. Build med_key_to_token_id from header_to_ind
# Valid keys start with "Medication:" and end with a number
med_key_to_token_id = {}
for header, idx in header_to_ind.items():
    if str(header).startswith("Medication:"):
        try:
            # Extract the key at the end (e.g., "Medication:NAME 123" -> 123)
            key_str = str(header).rsplit(' ', 1)[-1]
            key_int = int(key_str)
            med_key_to_token_id[key_int] = idx
        except ValueError:
            # Some headers might not match the format perfectly, skip them
            continue

print(f"Found {len(med_key_to_token_id)} Medication keys in header_to_ind")

# 2. Build rxnorm_to_med_key with conflict resolution
print("Building optimized RxNorm mapping...")

# Filter for RxNorm
rxnorm_map_df = nyu_medication_mapping[nyu_medication_mapping['type'] == 'RxNorm'].copy()
rxnorm_map_df['code_str'] = rxnorm_map_df['code'].astype(str)

# We want to map code_str -> single med_key
# But first, filter to med_keys that exist in med_key_to_token_id
valid_med_keys = set(med_key_to_token_id.keys())

# Group by RxNorm code to see all possible VALID medication keys
# Create a mapping: RxNorm Code -> List of Valid Medication Keys
code_to_valid_keys = {}

# Optimized grouping
# Filter rows first
valid_rows = rxnorm_map_df[rxnorm_map_df['medicationkey'].isin(valid_med_keys)]

for code, group in valid_rows.groupby('code_str'):
    keys = group['medicationkey'].unique()
    code_to_valid_keys[code] = keys

# Resolve conflicts using token frequency
rxnorm_to_med_key = {}
resolved_counts = 0
total_conflicts = 0

for code, keys in code_to_valid_keys.items():
    if len(keys) == 1:
        rxnorm_to_med_key[code] = keys[0]
    else:
        # Multiple valid tokens for this RxNorm code. Pick the most frequent one.
        total_conflicts += 1
        best_key = None
        max_count = -1
        
        for k in keys:
            token_id = med_key_to_token_id[k]
            # Use computed token_counts
            count = token_counts.get(token_id, 0)
            if count > max_count or (count == max_count and best_key is None):
                max_count = count
                best_key = k
        
        rxnorm_to_med_key[code] = best_key
        resolved_counts += 1

print(f"Mapped {len(rxnorm_to_med_key)} RxNorm codes to unique medication keys.")
print(f"Resolved {total_conflicts} conflicts using frequency data.")

# 3. Create mapping from full ehrshot code string to token_id
# We only care about codes starting with "RxNorm/"
rxnorm_codes_ehrshot = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('RxNorm/'), 'code'].unique()

count_mapped = 0
rxnorm_translation_map = {}

for full_code in rxnorm_codes_ehrshot:
    # Extract code part: "RxNorm/123" -> "123"
    code_val = full_code.split('/')[1]
    
    if code_val in rxnorm_to_med_key:
        med_key = rxnorm_to_med_key[code_val]
        if med_key in med_key_to_token_id:
            token_id = med_key_to_token_id[med_key]
            rxnorm_translation_map[full_code] = token_id
            count_mapped += 1

print(f"Mapped {count_mapped} out of {len(rxnorm_codes_ehrshot)} unique RxNorm codes found in EHRSHOT to target tokens.")

# 4. Apply the translation to the dataframe
# Re-use the 'target_token_id' column we created earlier, filling in NaNs where applicable

# Find rows with RxNorm codes that we can translate
mask_rx = ehrshotcsv['code'].isin(rxnorm_translation_map.keys())

# Map directly to token_id (since our map values are already token_ids)
ehrshotcsv.loc[mask_rx, 'target_token_id'] = ehrshotcsv.loc[mask_rx, 'code'].map(rxnorm_translation_map)

print(f"Updated {mask_rx.sum()} rows with Medication translations.")
print(ehrshotcsv.loc[mask_rx, ['code', 'target_token_id']].head())

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

In [ ]:
import pandas as pd

# 1. Keep only RxNorm rows from your events table
rx_events = ehrshotcsv[ehrshotcsv["code"].str.startswith("RxNorm/")].copy()

# Extract the numeric/string RxNorm code part after "RxNorm/"
rx_events["rxnorm_code"] = rx_events["code"].str.split("/").str[1]

# 2. Keep only RxNorm rows from your mapping table
rx_map = nyu_medication_mapping[nyu_medication_mapping["type"] == "RxNorm"].copy()

# Make sure both sides are strings so merge works even if some codes look like 10370-171-09 or C08C
rx_events["rxnorm_code"] = rx_events["rxnorm_code"].astype(str)
rx_map["code"] = rx_map["code"].astype(str)

# 3. Merge to see which events can be translated
rx_events_merged = rx_events.merge(
    rx_map[["code", "medicationkey", "name"]],
    left_on="rxnorm_code",
    right_on="code",
    how="left",
    suffixes=("", "_map"),
)

# 4. Coverage at the *event* level
n_events = len(rx_events_merged)
n_mapped_events = rx_events_merged["medicationkey"].notna().sum()
event_coverage = n_mapped_events / n_events if n_events > 0 else float("nan")

print(f"RxNorm events:        {n_events:,}")
print(f"Mapped events:        {n_mapped_events:,} ({event_coverage:.2%})")

# 5. Coverage at the *unique code* level
used_codes = rx_events_merged["rxnorm_code"].unique()
n_codes = len(used_codes)

mapped_codes = rx_map[rx_map["code"].isin(used_codes)]["code"].unique()
n_mapped_codes = len(mapped_codes)
code_coverage = n_mapped_codes / n_codes if n_codes > 0 else float("nan")

print(f"Unique RxNorm codes:  {n_codes:,}")
print(f"Mapped unique codes:  {n_mapped_codes:,} ({code_coverage:.2%})")

# 6. (Optional) peek at some unmapped codes
unmapped_codes = sorted(set(used_codes) - set(mapped_codes))
print("Example unmapped codes:", unmapped_codes[:20])

In [ ]:
# Analyze top 20 most frequent RxNorm codes
top_n = 20
rx_counts = ehrshotcsv.loc[ehrshotcsv['code'].str.startswith('RxNorm/'), 'code'].value_counts().head(top_n)

# Prepare a list to store details
results = []

for code_full, count in rx_counts.items():
    # Check mapping status
    is_mapped = code_full in rxnorm_translation_map
    
    # Attempt to retrieve the medication name
    # code_full format is "RxNorm/{code}"
    code_val = code_full.split('/')[1]
    
    # Look up name in the mapping dataframe (loaded in previous cells)
    name_matches = nyu_medication_mapping.loc[
        (nyu_medication_mapping['type'] == 'RxNorm') & 
        (nyu_medication_mapping['code'].astype(str) == code_val),
        'name'
    ]
    
    name = name_matches.iloc[0] if not name_matches.empty else "Unknown"
    
    results.append({
        "EHRSHOT Code": code_full,
        "Name": name,
        "Frequency": count,
        "Mapped": is_mapped
    })

# Display results
top_rx_df = pd.DataFrame(results)
print(f"Top {top_n} Most Frequent RxNorm Codes in EHRSHOT data:")
display(top_rx_df)

In [ ]:
rxnorm_to_med_key['313782']

In [ ]:
nyu_medication_mapping[nyu_medication_mapping['code'] == '313782']

## Diagnosis

### ICD-O-3

In [ ]:
# 1. Build diag_code_to_token_id from header_to_ind
diag_code_to_token_id = {}
duplicate_diag_codes = 0

for header, idx in header_to_ind.items():
    if str(header).startswith("Diagnosis:"):
        # Assumption: The code is the last part of the string separated by space
        code_part = str(header).split(' ')[-1]
        
        # Determine if we should handle duplicates
        # Ideally, codes should be unique or we loop through and overwrite/ignore.
        # Let's map code -> token_id
        if code_part in diag_code_to_token_id:
            duplicate_diag_codes += 1
            pass 
        
        diag_code_to_token_id[code_part] = idx

print(f"Found {len(diag_code_to_token_id)} Diagnosis codes in header_to_ind (Duplicates skipped/overwritten: {duplicate_diag_codes})")

# 2. Build icdo3_translation_map
icdo3_codes_ehrshot = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('ICDO3/'), 'code'].unique()

print(f"Found {len(icdo3_codes_ehrshot)} unique ICDO3-related codes in EHRSHOT")

icdo3_translation_map = {}
mapped_count_icd = 0

for full_code in icdo3_codes_ehrshot:
    full_code_str = str(full_code)
    
    # Logic to extract code:
    # If '-' exists, take part after last '-'
    # Else take part after last '/'
    if '-' in full_code_str:
        extracted_code = full_code_str.split('-')[-1]
    else:
        extracted_code = full_code_str.split('/')[-1]
        
    if extracted_code in diag_code_to_token_id:
        token_id = diag_code_to_token_id[extracted_code]
        icdo3_translation_map[full_code] = token_id
        mapped_count_icd += 1

print(f"Mapped {mapped_count_icd} out of {len(icdo3_codes_ehrshot)} unique ICDO3 codes to target tokens.")

# 3. Apply the translation
mask_icd = ehrshotcsv['code'].isin(icdo3_translation_map.keys())
ehrshotcsv.loc[mask_icd, 'target_token_id'] = ehrshotcsv.loc[mask_icd, 'code'].map(icdo3_translation_map)

print(f"Updated {mask_icd.sum()} rows with ICDO3 translations.")
print(ehrshotcsv.loc[mask_icd, ['code', 'target_token_id']].head())

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

### SNOMED

In [ ]:
# read in tsv file
snomed_icd_mapping = pd.read_csv("mapping/tls_Icd10cmHumanReadableMap_US1000124_20250901.tsv", sep="\t")

In [ ]:
# 1. Prepare snomed_icd_mapping lookup
# Filter for active mappings and mapGroup 1
active_map = snomed_icd_mapping[(snomed_icd_mapping['active'] == 1) & (snomed_icd_mapping['mapGroup'] == 1)].copy()
active_map = active_map.sort_values(by=['referencedComponentId', 'mapPriority'])
active_map = active_map.drop_duplicates(subset=['referencedComponentId'], keep='last')

# Create dict: SNOMED Code (int) -> ICD 10 Code (str)
snomed_int_to_icd = dict(zip(active_map['referencedComponentId'], active_map['mapTarget']))

print(f"Created mapping dictionary with {len(snomed_int_to_icd)} SNOMED entries (using mapGroup=1, last mapPriority).")

# 2. Build SNOMED -> Token ID Map
snomed_translation_map = {}
mapped_count_snomed = 0

# diagnosis codes already in diag_code_to_token_id from previous step
# diag_code_to_token_id keys are ICD10 codes (e.g. "C56.9")

snomed_codes_in_ehr_unique = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('SNOMED/'), 'code'].unique()

for full_code in snomed_codes_in_ehr_unique:
    # Full code: e.g., "SNOMED/223003"
    snomed_str = full_code.split('/')[-1]
    
    try:
        snomed_int = int(snomed_str)
    except ValueError:
        continue
        
    # Get mapped ICD
    if snomed_int in snomed_int_to_icd:
        icd_target = snomed_int_to_icd[snomed_int]
        # mapTarget might be non-string or NaN, ensure string
        if pd.isna(icd_target):
            continue
        icd_target = str(icd_target)
        
        # Try finding in our header mapping
        if icd_target in diag_code_to_token_id:
            token_id = diag_code_to_token_id[icd_target]
            snomed_translation_map[full_code] = token_id
            mapped_count_snomed += 1

print(f"Mapped {mapped_count_snomed} out of {len(snomed_codes_in_ehr_unique)} unique SNOMED codes to target tokens.")

# 3. Apply translations (Fill only where currently NaN to avoid overwriting previous work, 
# though SNOMED rows likely stem from different rows than Race/Rx/ICDO3)
mask_snomed = ehrshotcsv['code'].isin(snomed_translation_map.keys())

# We can overwrite or fillna. Since code column is unique per row, overwrite is fine.
ehrshotcsv.loc[mask_snomed, 'target_token_id'] = ehrshotcsv.loc[mask_snomed, 'code'].map(snomed_translation_map)

print(f"Updated {mask_snomed.sum()} rows with SNOMED translations.")
print(ehrshotcsv.loc[mask_snomed, ['code', 'target_token_id']].head())

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

## Labs

In [ ]:
import pickle
stats_dict_path = os.path.join(NYU_DATA_DIR, 'lab_stats.data')
with open(stats_dict_path, 'rb') as file:
    stats_dict = pickle.load(file)
stats_dict


In [ ]:
# 1. Parse header_to_ind to build lookup structures
# Structure: (LOINC, BinIndex) -> TokenID
# Also need: LOINC -> LabFeatureName (to look up stats)

loinc_bin_to_token = {}
loinc_to_feature_name = {}

# Regex to parse header:
# Start with "Lab:"
# Capture Feature Name until start of bracket "[normedValue"
# Capture Bin range "between_X,Y]"
# Example: Lab:Cancer Ag 125 [Units/volume] in Serum or Plasma 10334-1[normedValue_sd_between_-10,-3]
# Feature Name: Lab:Cancer Ag 125 [Units/volume] in Serum or Plasma 10334-1
# Bin: -10, -3 (corresponds to index 0)

# Bins from user function: -10, -3, -1, -0.5, 0.5, 1, 3, 10
# Bin string map:
# -10,-3 -> idx 0
# -3,-1  -> idx 1
# -1,-0.5 -> idx 2
# -0.5,0.5 -> idx 3
# 0.5,1 -> idx 4
# 1,3 -> idx 5
# 3,10 -> idx 6

bin_str_to_idx = {
    "-10,-3": 0,
    "-3,-1": 1,
    "-1,-0.5": 2,
    "-0.5,0.5": 3,
    "0.5,1": 4,
    "1,3": 5,
    "3,10": 6
}

# Regex explanation:
# ^(Lab:.+) -> Capture starts with Lab: and everything until [
# \[normedValue_sd_between_ -> literal
# (.*) -> capture bin string e.g. -10,-3
# \]$ -> end with ]
pattern = re.compile(r"^(Lab:.+)\[normedValue_sd_between_(.*)\]$")

for header, token_id in header_to_ind.items():
    header_str = str(header)
    if not header_str.startswith("Lab:"):
        continue
    
    match = pattern.match(header_str)
    if match:
        feature_name = match.group(1)
        bin_str = match.group(2)
        
        # Extract LOINC from feature name (usually last word)
        # e.g. "... Plasma 10334-1" -> "10334-1"
        try:
            loinc_code = feature_name.split(' ')[-1]
        except:
            continue
            
        if bin_str in bin_str_to_idx:
            bin_idx = bin_str_to_idx[bin_str]
            
            # Store mappings
            loinc_bin_to_token[(loinc_code, bin_idx)] = token_id
            loinc_to_feature_name[loinc_code] = feature_name

print(f"Parsed {len(loinc_to_feature_name)} unique Lab features from header_to_ind.")

# 2. Process EHRSHOT Lab data
# Filter rows with LOINC codes
ehrshot_labs = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('LOINC')].copy()

# Extract LOINC code without prefix
# "LOINC/123-4" -> "123-4"
ehrshot_labs['loinc_extracted'] = ehrshot_labs['code'].astype(str).apply(lambda x: x.split('/')[1])

# Filter only those we have mappings for
ehrshot_labs = ehrshot_labs[ehrshot_labs['loinc_extracted'].isin(loinc_to_feature_name)]

print(f"Found {len(ehrshot_labs)} rows with LOINC codes matching our target features.")

if len(ehrshot_labs) == 0:
    print("No matching Lab rows found. Check LOINC formats.")
else:
    # 3. Normalize and Discretize
    # We need to vectorized this or loop efficiently.
    # Group by LOINC code to apply same stats
    
    # Pre-fetch stats for all relevant LOINCs
    # LOINC -> (mean, std)
    # stats_dict keys are feature_names
    
    loinc_stats = {}
    for loinc, feat_name in loinc_to_feature_name.items():
        if feat_name in stats_dict:
            loinc_stats[loinc] = stats_dict[feat_name]
        else:
            # If not in stats_dict, maybe we skip or assume 0,1?
            # User said "stats_dict of our data is already loaded"
            pass

    # Bins for digitize
    # bins = [-10, -3, -1, -0.5, 0.5, 1, 3, 10]
    # np.searchsorted or digitize returns index.
    # Logic in user code: (val >= bins[i]) & (val < bins[i+1])
    # intervals: [-10, -3), [-3, -1), [-1, -0.5), [-0.5, 0.5), [0.5, 1), [1, 3), [3, 10)
    # indices: 0, 1, 2, 3, 4, 5, 6
    # Note: user code uses < bins[i+1] which suggests right-open intervals.
    # The last bin is [3, 10). What about >= 10? User code seems to stop at 10.
    # We'll use np.digitize. 
    # bins array for digitize: [-10, -3, -1, -0.5, 0.5, 1, 3, 10]
    # result 1 => between -10 and -3? No, digitize usage depends on 'right'.
    # default right=False: bins[i-1] <= x < bins[i]
    # We want: 
    # 0: -10 <= x < -3
    # 1: -3 <= x < -1
    # ...
    # This matches digitize with bins=[-10, -3, -1, -0.5, 0.5, 1, 3, 10]
    # BUT digitize returns 1 for lowest bin. We want 0-indexed for our map.
    # So we'll subtract 1.
    
    bin_boundaries = np.array([-10, -3, -1, -0.5, 0.5, 1, 3, 10])
    
    # We'll process in groups
    def process_group(group):
        loinc = group['loinc_extracted'].iloc[0]
        if loinc not in loinc_stats:
            return None # Skip if no stats
            
        mean, std = loinc_stats[loinc]
        
        # Avoid division by zero
        if std == 0:
            std = 1.0
            
        # Normalize
        vals = pd.to_numeric(group['value'], errors='coerce').fillna(0) # Ensure numeric
        norm_vals = (vals - mean) / std
        
        # Discretize
        # We need to handle values outside range [-10, 10]
        # values < -10 will get index 0 (if we clip) or separate?
        # User code: np.array((val >= bins[i]) & (val < bins[i+1]))
        # This implies strict bounds. Values outside [-10, 10] are ignored (dropped).
        
        # We find indices using searchsorted to match user explicitly or digitize
        # Using digitize:
        # inds = np.digitize(norm_vals, bin_boundaries)
        # inds=1 -> x < -3 (if bins start at -10? No)
        # Let's use strict logic:
        
        bin_indices = np.full(len(norm_vals), -1)
        
        for i in range(len(bin_boundaries) - 1):
            low = bin_boundaries[i]
            high = bin_boundaries[i+1]
            mask = (norm_vals >= low) & (norm_vals < high)
            bin_indices[mask] = i
            
        # Create result series
        token_ids = pd.Series(np.nan, index=group.index)
        
        # Map valid indices to tokens
        # Valid indices are 0 to 6
        valid_mask = bin_indices != -1
        
        if valid_mask.any():
            # Apply mapping for this LOINC
            # dict lookup: (loinc, idx) -> token
            # Doing row by row or map?
            # loinc is constant for this group.
            
            # Map index to token
            # We can use a small lookup for this group
            group_tokens = {}
            for i in range(7):
                if (loinc, i) in loinc_bin_to_token:
                    group_tokens[i] = loinc_bin_to_token[(loinc, i)]
            
            # Map the indices
            # Only map valid ones
            mapped = pd.Series(bin_indices[valid_mask]).map(group_tokens)
            token_ids.loc[valid_mask] = mapped.values
            
        return token_ids

    # Apply processing
    # groupby apply might be slow if many groups. 
    # But we have 10M rows... let's see unique LOINCs count.
    # If small number of LOINCs, groupby is fine.
    
    unique_loincs_count = ehrshot_labs['loinc_extracted'].nunique()
    print(f"Processing {unique_loincs_count} unique LOINC codes...")
    
    # We can optimize by iterating groups manually
    results = []
    
    # We need to update original ehrshotcsv
    # We can create a series "lab_token_ids" aligned with ehrshot_labs index
    
    lab_token_ids = ehrshot_labs.groupby('loinc_extracted', group_keys=False).apply(process_group)

    # 4. Update DataFrame
    # lab_token_ids index should align with ehrshotcsv index
    # But groupby apply result might be tricky if it returns None.
    # Filter out NaNs
    lab_token_ids = lab_token_ids.dropna()
    
    print(f"Mapped {len(lab_token_ids)} Lab rows to tokens.")
    
    # Assign to target_token_id
    ehrshotcsv.loc[lab_token_ids.index, 'target_token_id'] = lab_token_ids

    print(f"Total rows updated with Lab translations: {len(lab_token_ids)}")
    print(ehrshotcsv.loc[lab_token_ids.index[:5], ['code', 'value', 'target_token_id']].head())

In [ ]:
# 1. Parse header_to_ind to build lookup structures
# Structure: (LOINC, BinIndex) -> TokenID
# Also need: LOINC -> LabFeatureName (to look up stats)

loinc_bin_to_token = {}
loinc_to_feature_name = {}

# Regex to parse header:
# Start with "Lab:"
# Capture Feature Name until start of bracket "[normedValue"
# Capture Bin range "between_X,Y]"
# Example: Lab:Cancer Ag 125 [Units/volume] in Serum or Plasma 10334-1[normedValue_sd_between_-10,-3]
# Feature Name: Lab:Cancer Ag 125 [Units/volume] in Serum or Plasma 10334-1
# Bin: -10, -3 (corresponds to index 0)

# Bins from user function: -10, -3, -1, -0.5, 0.5, 1, 3, 10
# Bin string map:
# -10,-3 -> idx 0
# -3,-1  -> idx 1
# -1,-0.5 -> idx 2
# -0.5,0.5 -> idx 3
# 0.5,1 -> idx 4
# 1,3 -> idx 5
# 3,10 -> idx 6

bin_str_to_idx = {
    "-10,-3": 0,
    "-3,-1": 1,
    "-1,-0.5": 2,
    "-0.5,0.5": 3,
    "0.5,1": 4,
    "1,3": 5,
    "3,10": 6
}

# Regex explanation:
# ^(Lab:.+) -> Capture starts with Lab: and everything until [
# \[normedValue_sd_between_ -> literal
# (.*) -> capture bin string e.g. -10,-3
# \]$ -> end with ]
pattern = re.compile(r"^(Lab:.+)\[normedValue_sd_between_(.*)\]$")

for header, token_id in header_to_ind.items():
    header_str = str(header)
    if not header_str.startswith("Lab:"):
        continue
    
    match = pattern.match(header_str)
    if match:
        feature_name = match.group(1)
        bin_str = match.group(2)
        
        # Extract LOINC from feature name (usually last word)
        # e.g. "... Plasma 10334-1" -> "10334-1"
        try:
            loinc_code = feature_name.split(' ')[-1]
        except:
            continue
            
        if bin_str in bin_str_to_idx:
            bin_idx = bin_str_to_idx[bin_str]
            
            # Store mappings
            loinc_bin_to_token[(loinc_code, bin_idx)] = token_id
            loinc_to_feature_name[loinc_code] = feature_name

print(f"Parsed {len(loinc_to_feature_name)} unique Lab features from header_to_ind.")

# 2. Process EHRSHOT Lab data
# Filter rows with LOINC codes
ehrshot_labs = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('LOINC')].copy()

# Extract LOINC code without prefix
# "LOINC/123-4" -> "123-4"
ehrshot_labs['loinc_extracted'] = ehrshot_labs['code'].astype(str).apply(lambda x: x.split('/')[1])

# Filter only those we have mappings for
ehrshot_labs = ehrshot_labs[ehrshot_labs['loinc_extracted'].isin(loinc_to_feature_name)]

print(f"Found {len(ehrshot_labs)} rows with LOINC codes matching our target features.")

if len(ehrshot_labs) == 0:
    print("No matching Lab rows found. Check LOINC formats.")
else:
    # 3. Normalize and Discretize
    
    # === CONFIGURATION ===
    # Set to True to calculate mean/std from EHRSHOT data (good if units differ)
    # Set to False to use the pre-loaded stats_dict (from training data)
    USE_OWN_STATS = True 
    # =====================

    # Pre-fetch stats for all relevant LOINCs if using external
    loinc_stats = {}
    if not USE_OWN_STATS:
        for loinc, feat_name in loinc_to_feature_name.items():
            if feat_name in stats_dict:
                loinc_stats[loinc] = stats_dict[feat_name]
        print("Using EXTERNAL statistics from stats_dict.")
    else:
        print("Using INTERNAL statistics calculated from EHRSHOT data.")
    
    bin_boundaries = np.array([-10, -3, -1, -0.5, 0.5, 1, 3, 10])
    
    # We'll process in groups
    def process_group(group):
        loinc = group['loinc_extracted'].iloc[0]
        
        # Parse values
        vals_numeric = pd.to_numeric(group['value'], errors='coerce')
        
        if USE_OWN_STATS:
             # Calculate stats from this group's data
             mean = vals_numeric.mean()
             std = vals_numeric.std()
             
             if pd.isna(mean): mean = 0.0
             if pd.isna(std): std = 1.0
        else:
            if loinc not in loinc_stats:
                return None # Skip if no stats
            mean, std = loinc_stats[loinc]
        
        # Avoid division by zero
        if std == 0:
            std = 1.0
            
        # Normalize
        # Fill NaNs with 0 (consistent with previous logic)
        vals = vals_numeric.fillna(0) 
        norm_vals = (vals - mean) / std
        
        # Discretize
        # We need to handle values outside range [-10, 10]
        
        bin_indices = np.full(len(norm_vals), -1)
        
        for i in range(len(bin_boundaries) - 1):
            low = bin_boundaries[i]
            high = bin_boundaries[i+1]
            mask = (norm_vals >= low) & (norm_vals < high)
            bin_indices[mask] = i
            
        # Create result series
        token_ids = pd.Series(np.nan, index=group.index)
        
        # Map valid indices to tokens
        # Valid indices are 0 to 6
        valid_mask = bin_indices != -1
        
        if valid_mask.any():
            # Apply mapping for this LOINC
            # dict lookup: (loinc, idx) -> token
            # We can use a small lookup for this group
            group_tokens = {}
            for i in range(7):
                if (loinc, i) in loinc_bin_to_token:
                    group_tokens[i] = loinc_bin_to_token[(loinc, i)]
            
            # Map the indices
            mapped = pd.Series(bin_indices[valid_mask]).map(group_tokens)
            token_ids.loc[valid_mask] = mapped.values
            
        return token_ids

    # Apply processing
    unique_loincs_count = ehrshot_labs['loinc_extracted'].nunique()
    print(f"Processing {unique_loincs_count} unique LOINC codes...")
    
    # Calculate lab_token_ids
    lab_token_ids = ehrshot_labs.groupby('loinc_extracted', group_keys=False).apply(process_group)

    # 4. Update DataFrame
    # Filter out NaNs
    lab_token_ids = lab_token_ids.dropna()
    
    print(f"Mapped {len(lab_token_ids)} Lab rows to tokens.")
    
    # Assign to target_token_id
    ehrshotcsv.loc[lab_token_ids.index, 'target_token_id'] = lab_token_ids

    print(f"Total rows updated with Lab translations: {len(lab_token_ids)}")
    print(ehrshotcsv.loc[lab_token_ids.index[:5], ['code', 'value', 'target_token_id']].head())

In [ ]:
# Analyze top 20 most frequent LOINC codes
top_n = 20
# Ensure we are looking at LOINC codes
loinc_counts = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('LOINC'), 'code'].value_counts().head(top_n)

results_loinc = []

for code_full, count in loinc_counts.items():
    # Extract LOINC part: "LOINC/123-4" -> "123-4"
    try:
        loinc_extracted = str(code_full).split('/')[1]
    except IndexError:
        loinc_extracted = str(code_full)

    # Check mapping status in header
    # loinc_to_feature_name was created in the previous cell
    is_in_header = loinc_extracted in loinc_to_feature_name
    
    # Get Feature Name
    feature_name = loinc_to_feature_name.get(loinc_extracted, "Unknown / Not in Header")
    
    # Check if stats exist for this feature
    # stats_dict keys are feature names
    has_stats = False
    if 'stats_dict' in locals() and feature_name != "Unknown / Not in Header":
        has_stats = feature_name in stats_dict

    # "Fully Mapped" implies it's in the header AND we have stats for it
    is_fully_mapped = is_in_header and has_stats

    results_loinc.append({
        "EHRSHOT Code": code_full,
        "LOINC Extracted": loinc_extracted,
        "Feature Name": feature_name,
        "Frequency": count,
        "In Header": is_in_header,
        "Has Stats": has_stats,
        "Fully Mapped": is_fully_mapped
    })

# Display results
top_loinc_df = pd.DataFrame(results_loinc)
print(f"Top {top_n} Most Frequent LOINC Codes in EHRSHOT data:")
display(top_loinc_df)

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

In [ ]:
stats_dict['Lab:Body temperature 8310-5']

In [ ]:
# get mean of LOINC/8310-5

# filter
subset = ehrshotcsv[ehrshotcsv['code'] == 'LOINC/8310-5'].copy()

# coerce string "NA" to NaN and convert to numeric
subset['value'] = pd.to_numeric(subset['value'], errors='coerce')

# calculate mean and std
mean_val = subset['value'].mean()
std_val = subset['value'].std()

print(f"Mean: {mean_val}")
print(f"Std: {std_val}")

## End

In [ ]:
# 1. Filter out rows with NA for target_token_id
ehrshot_clean = ehrshotcsv.dropna(subset=['target_token_id'])

# Save to EHRSHOT_Data/ehrshot_mapped.csv
output_dir = 'EHRSHOT_Data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

mapped_path = os.path.join(output_dir, 'ehrshot_mapped.csv')
ehrshot_clean.to_csv(mapped_path, index=False)
print(f"Saved cleaned mapped data to {mapped_path}. Row count: {len(ehrshot_clean)}")

# 2. Create subset with specific columns
subset_cols = ['patient_id', 'visit_id', 'start', 'target_token_id']
ehrshot_final = ehrshot_clean[subset_cols]

# Save to EHRSHOT_Data/ehrshot_token_ids.csv
tokens_path = os.path.join(output_dir, 'ehrshot_token_ids.csv')
ehrshot_final.to_csv(tokens_path, index=False)
print(f"Saved final tokens data to {tokens_path}. Row count: {len(ehrshot_final)}")
print("First 5 rows of final tokens:\n", ehrshot_final.head())

In [ ]:
# load ehrshot_final
output_dir = 'EHRSHOT_Data'
tokens_path = os.path.join(output_dir, 'ehrshot_token_ids.csv')
ehrshot_final = pd.read_csv(tokens_path)

# Building Eval Dataset

In [ ]:
# 1. Identify Demographics Tokens
demo_target_strings = set(ehrshot_to_target_map.values())
demo_token_ids = set()
for t_str in demo_target_strings:
    if t_str in header_to_ind:
        demo_token_ids.add(header_to_ind[t_str])

print(f"Identified {len(demo_token_ids)} demographics token IDs.")

In [ ]:
# 2. Prepare Data
# Ensure datetime
ehrshot_final['start'] = pd.to_datetime(ehrshot_final['start'])

# Split into Demo and Clinical
# We'll assume demographics tokens are EXACTLY those in our set.
# Note: ehrshot_final['target_token_id'] might need to be int.
ehrshot_final['target_token_id'] = pd.to_numeric(ehrshot_final['target_token_id'], errors='coerce')
ehrshot_final = ehrshot_final.dropna(subset=['target_token_id'])
ehrshot_final['target_token_id'] = ehrshot_final['target_token_id'].astype(int)

is_demo = ehrshot_final['target_token_id'].isin(demo_token_ids)
demo_df = ehrshot_final[is_demo].copy()
clinical_df = ehrshot_final[~is_demo].copy()
# drop token_id = 26836
age_df = ehrshot_final[ehrshot_final['target_token_id'] == 26836]
clinical_df = clinical_df[clinical_df['target_token_id'] != 26836]

print(f"Demographics rows: {len(demo_df)}")
print(f"Age rows: {len(age_df)}")
print(f"Clinical rows: {len(clinical_df)}")

# 3. Group and Process
# Pre-compute demographics for each patient: patient_id -> list of tokens
demo_grp = demo_df.groupby('patient_id')['target_token_id'].apply(list)
patient_demo_tokens = demo_grp.to_dict()

In [ ]:
clinical_df[clinical_df['visit_id'].isna()]

In [ ]:
# RE-PROCESS: Merge visits on the same day & Remove duplicates
from collections import defaultdict

print("Refining visits to day-level...")

# 1. Day-level grouping
# clinical_df comes from previous step
clinical_df['start_date'] = clinical_df['start'].dt.floor('D')

# Group by patient and day
# We aggregate tokens into a list, then deduplicate
print("Grouping by Patient and Date...")
day_groups = clinical_df.groupby(['patient_id', 'start_date'])['target_token_id'].apply(list)

# Deduplicate tokens within each day-visit
# We use set to remove duplicates, then convert back to list
# Sorting ensures deterministic order which is often good
print("Deduplicating tokens...")
day_groups = day_groups.apply(lambda x: sorted(list(set(x))))

# 2. Build Timelines
# Convert series to similar structure as before: dict of list of dicts
# patient_id -> [{'time': date, 'tokens': [...]}, ...]
print("Building timelines...")

patient_timelines = defaultdict(list)

# Iterate over the series
# The index is (patient_id, start_date)
# We can reset index to iterate easier or iterate items
day_groups_df = day_groups.reset_index()
day_groups_df = day_groups_df.sort_values(['patient_id', 'start_date'])

for row in day_groups_df.itertuples(index=False):
    # row has patient_id, start_date, target_token_id
    patient_timelines[row.patient_id].append({
        'time': row.start_date,
        'tokens': row.target_token_id
    })

print(f"Built timelines for {len(patient_timelines)} patients.")

In [ ]:
# 26836 birth
ind_to_header[42299]

In [ ]:
ehrshot_final[ehrshot_final['patient_id'] == 115967111]

In [ ]:
patient_timelines[115967111]

In [ ]:
header_to_ind['demographics_age[between_60_and_65]']

In [ ]:
def get_age_bucket_header(age_years):
    if 1 <= age_years < 5: return "demographics_age[between_1_and_5]"
    if 5 <= age_years < 10: return "demographics_age[between_5_and_10]"
    if 10 <= age_years < 20: return "demographics_age[between_10_and_20]"
    if 20 <= age_years < 30: return "demographics_age[between_20_and_30]"
    if 30 <= age_years < 40: return "demographics_age[between_30_and_40]"
    if 40 <= age_years < 50: return "demographics_age[between_40_and_50]"
    if 50 <= age_years < 60: return "demographics_age[between_50_and_60]"
    if 60 <= age_years < 65: return "demographics_age[between_60_and_65]"
    if 65 <= age_years < 70: return "demographics_age[between_65_and_70]"
    if 70 <= age_years < 75: return "demographics_age[between_70_and_75]"
    if 75 <= age_years < 80: return "demographics_age[between_75_and_80]"
    if 80 <= age_years < 85: return "demographics_age[between_80_and_85]"
    if 85 <= age_years < 90: return "demographics_age[between_85_and_90]"
    if 90 <= age_years < 95: return "demographics_age[between_90_and_95]"
    if 95 <= age_years < 100: return "demographics_age[between_95_and_100]"
    if 100 <= age_years < 150: return "demographics_age[between_100_and_150]"
    return None

In [ ]:
# 4. Create Examples for Label DF
print("Creating inference examples...")

# Pre-load label_df if not already (it is in memory as label_df)
# Ensure timestamps
disease = "new_pancan"
label_df = pd.read_parquet(os.path.join(EHRSHOT_MEDS_DIR, f'labels/{disease}/labels.parquet'))
label_df['prediction_time'] = pd.to_datetime(label_df['prediction_time'])

# Prepare DOB map
# Ensure 'start' is datetime
age_df['start'] = pd.to_datetime(age_df['start'], errors='coerce')
dobs = age_df.set_index('patient_id')['start'].to_dict()

inference_data = []

count_no_history = 0

# Iterate labels
for _, row in label_df.iterrows():
    pid = row['subject_id']
    pred_time = row['prediction_time']
    label = row['boolean_value']
    
    if pid not in patient_timelines:
        count_no_history += 1
        continue
        
    timeline = patient_timelines[pid]
    
    # Filter visits <= pred_time
    valid_visits = [v for v in timeline if v['time'] <= pred_time]
    
    if not valid_visits:
        count_no_history += 1
        continue
    
    # Get DOB
    dob = dobs.get(pid, pd.NaT)
    
    # Process valid visits to add age tokens (to ALL visits) and demographics (to FIRST visit)
    processed_valid_visits = []
    
    for i, v in enumerate(valid_visits):
        # Create a shallow copy of the visit dict so we don't modify the original timeline
        current_visit = v.copy()
        
        # Get tokens list (ensure it's a new list)
        current_tokens = list(current_visit['tokens'])
        
        # 1. Add Age Token (Dynamic, for each visit)
        if not pd.isna(dob):
            # Calculate age at time of visit
            age_years = (current_visit['time'] - dob).days / 365.25
            # print(age_years)
            
            bucket_header = get_age_bucket_header(age_years)
            if bucket_header and bucket_header in header_to_ind:
                age_token_id = header_to_ind[bucket_header]
                # Prepend age token
                # print(age_token_id)
                current_tokens.insert(0, age_token_id)
        
        # 2. Add Static Demographics (Static, only for first visit)
        if i == 0 and pid in patient_demo_tokens:
            demo_tokens = patient_demo_tokens[pid]
            # Prepend demo tokens. Note: if age was added, it is at index 0.
            # We add demo tokens before everything (result: [Demo Tokens, Age Token, Original Tokens])
            current_tokens = demo_tokens + current_tokens
            
        current_visit['tokens'] = current_tokens
        processed_valid_visits.append(current_visit)
        
    # Use our processed list
    valid_visits = processed_valid_visits

    # Calculate relative days
    t0 = valid_visits[0]['time']
    
    processed_visits = []
    
    for v in valid_visits:
        # relative days (float or int)
        # using days component of timedelta
        diff = v['time'] - t0
        days = diff.days + diff.seconds / (24*3600) 
        
        processed_visits.append({
            'days': days,
            'tokens': v['tokens'] # list of ints
        })
        
    # Create entry
    entry = {
        'subject_id': pid,
        'prediction_time': pred_time,
        'boolean_value': label,
        'visits': processed_visits
    }
    
    inference_data.append(entry)

print(f"Generated {len(inference_data)} examples. (Skipped {count_no_history} due to no history)")

# 5. Save
output_disease_dir = os.path.join(output_dir, disease)
if not os.path.exists(output_disease_dir):
    os.makedirs(output_disease_dir)
output_path = os.path.join(output_disease_dir, 'ehrshot_inference_data.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(inference_data, f)
    
print(f"Saved inference data to {output_path}")


In [ ]:
inference_data[2]

In [ ]:
inference_data[2]

In [ ]:
import torch
from torch.utils.data import DataLoader
# Ensure condition_helper_ehrshot is importable
import sys
import os
sys.path.append(os.getcwd())
from condition_helper_ehrshot import SeqCLSDatasetEHRSHOT, aggregate_logits, GPTCONFIG_PARAMS
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
import numpy as np
from tqdm import tqdm

# 1. Define Evaluation Function (Single Process)
@torch.no_grad()
def evaluate_model(model, dataloader, device, condition_indices, condition_name="Pancreatic Cancer", 
                   agg_method="sum", threshold=0.5):
    """
    Evaluate the model on a dataset.
    """
    model.eval()
    all_logits = []
    all_labels = []
    
    # Get condition indices
    cond_ind = condition_indices
    if isinstance(condition_indices, dict):
        cond_ind = condition_indices[condition_name]
        
    # Assume output vocab size from model config or fallback
    try:
        vocab_size = model.config.vocab_size
    except:
        vocab_size = 57735  # Common vocab size in this project
        
    condition_mask = torch.zeros((vocab_size), dtype=torch.bool, device=device)
    # Filter indices that are out of bounds (just in case)
    valid_inds = [i for i in cond_ind if i < vocab_size]
    if len(valid_inds) < len(cond_ind):
        print(f"Warning: {len(cond_ind) - len(valid_inds)} condition indices ignored (>= vocab_size {vocab_size})")
        
    condition_mask[valid_inds] = True
    
    print(f"Evaluating on {len(dataloader.dataset)} examples...")
    
    for batch in tqdm(dataloader):
        # Move to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        # Forward pass using positional arguments as per eval_condition_mp.py
        # Args: idx, days_embed,, targets, pred_mask, pad_mask, attention_bias
        logits, _, _, _ = model(
            batch['token_arr'],
            batch['days_embed_arr'],
            None,
            batch['pred_mask'],
            batch['pad_mask'],
            batch['attention_bias'],
            patient_ids=batch.get('patient_ids', None)
        )
        
        # Get masked logits (predictions at the query position)
        masked_logits = logits[batch['pred_mask']]
        
        # Aggregate logits for the target condition specific tokens
        agg_logits = aggregate_logits(masked_logits, condition_mask, method=agg_method, dim=1)
        
        all_logits.append(agg_logits.cpu())
        all_labels.append(batch['labels'].cpu())
        
    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    
    # Compute Metrics
    # predictions = (all_logits >= threshold).astype(int)
    
    metrics = {
        'auroc': roc_auc_score(all_labels, all_logits),
        'auprc': average_precision_score(all_labels, all_logits)
    }
    
    return all_labels, all_logits, metrics


In [ ]:
from glob import glob
def load_latest_checkpoint(out_dir, device):
    """
    Load the latest checkpoint from the output directory.
    
    Args:
        out_dir: Directory containing checkpoints
        device: Device to load checkpoint to
        
    Returns:
        checkpoint: Loaded checkpoint
        checkpoint_model_args: Model arguments from checkpoint
        config: Configuration dictionary
    """
    checkpoint_files = glob(os.path.join(out_dir, 'ckpt-*.pt'))
    
    with open(os.path.join(out_dir, 'config.json'), 'r') as f:
        config = json.load(f)
    
    if not checkpoint_files:
        raise FileNotFoundError(f"No checkpoint files found in {out_dir}")
    
    checkpoint_files.sort(key=lambda x: int(x.split('-')[-1].split('.')[0]))
    
    latest_checkpoint = checkpoint_files[-1]
    
    print(f"Loading checkpoint: {latest_checkpoint}")
    
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    checkpoint_model_args = checkpoint['model_args']
    
    return checkpoint, checkpoint_model_args, config

In [ ]:
out_dir = os.path.join(MODEL_OUT_DIR, 'out_multi_win_True_standard_grad_clip_1.0_lr_0.00022_min_lr_2.2e-05_top_perc_0_n_layer_8_n_head_8_n_embd_512_rotary_embedding_True_use_xpos_False_agg_labels_False_reverse_temporal_decay_0.5_block_size_512')
condition_indices_path = os.path.join(NYU_PROCESSED_DIR, 'conditions2inds_ehrshot_plus_one.json')
with open(condition_indices_path, "r") as f:
    condition_indices = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Load model checkpoint
checkpoint, checkpoint_model_args, config = load_latest_checkpoint(out_dir, device)
print("Latest checkpoint loaded successfully")

# Create model
model_args = {}
for key in GPTCONFIG_PARAMS:
    if key in checkpoint_model_args:
        model_args[key] = checkpoint_model_args[key]

gptconf = GPTConfig(**model_args)
model = GPT(gptconf)

# Clean up state dict if needed (remove DDP-specific prefixes)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.to(device)

# Wrap model in DDP
model.eval()


In [ ]:
# 2. Setup Data Loading
disease = "new_hypertension" # "new_acutemi, new_pancan"
data_path = f'EHRSHOT_Data/{disease}/ehrshot_inference_data.pkl'

# Initialize Dataset
dataset = SeqCLSDatasetEHRSHOT(data_path, max_length=512, approach='direct', time_horizon=365)

# Initialize DataLoader
dataloader = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=None) # Default collate works for dicts of tensors

In [ ]:
with open(data_path, 'rb') as f:
    data = pickle.load(f)

In [ ]:
data[1]

In [ ]:
all_labels, all_logits, metrics = evaluate_model(model, dataloader, device, condition_indices, condition_name="Hypertension", 
                   agg_method="sum", threshold=0.5)

In [ ]:
metrics

In [ ]:
roc_auc_score(all_labels, 1-all_logits)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# def sigmoid(x):
#     x = np.asarray(x)
#     return 1 / (1 + np.exp(-x))

def plot_roc_pr(all_labels, all_logits, title_prefix=""):
    y_true = np.asarray(all_labels).astype(int).ravel()
    logits = np.asarray(all_logits).ravel()
    # y_score = sigmoid(logits)  # convert logits -> probabilities
    y_score = logits

    # --- ROC ---
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auroc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUROC = {auroc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix}ROC Curve".strip())
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- PR ---
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)

    # baseline = prevalence
    baseline = y_true.mean()

    plt.figure()
    plt.plot(recall, precision, label=f"AUPRC = {auprc:.4f}")
    plt.hlines(baseline, 0, 1, linestyle="--", label=f"Baseline = {baseline:.4f}")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix}Precision–Recall Curve".strip())
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return {"auroc": float(auroc), "auprc": float(auprc), "baseline": float(baseline)}

# Example:
# metrics = plot_roc_pr(all_labels, all_logits, title_prefix="Test | ")
# print(metrics)

In [ ]:
metrics = plot_roc_pr(all_labels, all_logits, title_prefix="Test | ")
print(metrics)